In [9]:
import sys
import os.path as osp
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm

repo_root = Path.cwd()
sys.path.insert(0, str(repo_root.parent / "hostApps" / "python" ))
from pdcv2_modules.pixMap import xy_map

In [10]:
root = Path("~/Desktop/UCSDPDCv2/data/").expanduser()
files = sorted(root.glob("Archive/TCR_5000ns*/**/**.csv"))

print(f"Running {len(files)} files")

Running 17 files


In [11]:
import matplotlib.pyplot as plt

pdc_layout = {
    0: (0, 0),
    1: (64, 0),
    2: (64, 64),
    3: (0, 64),
}
pdc_flipped = {0: False, 1: False, 2: True, 3: True}  # Some PDCs are flipped upside down relative

def read_data(file: Path | str):
    print(f"Running file {file}")
    df = pd.read_csv(file, delimiter=";")
    pdcs = sorted(int(c.removeprefix("SPAD_idx")) for c in df.columns if c.startswith("SPAD_idx"))
    unknown = [p for p in pdcs if p not in pdc_layout]
    assert not unknown, f"PDC(s) {unknown} in {file} have no entry in pdc_layout"
    return df, pdcs

def format_data(data: pd.DataFrame, pdcs: list[int]):
    grid = np.full((128, 128), np.nan)
    for pdc in pdcs:
        x_offset, y_offset = pdc_layout[pdc]
        for idx, val in zip(data[f"SPAD_idx{pdc}"], data[f"SPAD_TCR{pdc}"]):
            xy = xy_map(idx)
            if xy is not None:
                x, y = xy
                if pdc_flipped[pdc]:
                    x, y = 63 - x, 63 - y
                grid[y+y_offset, x+x_offset] = float(val)

    return grid

def draw_heatmaps(files: list[Path], save: str|Path|bool = False, show: bool = True,
                  vrange: tuple[float | None, float | None] | None = None):
    vmin, vmax = vrange if vrange is not None else (None, None)
    cmap = plt.get_cmap().copy()
    cmap.set_bad("white")  # NaN (missing PDCs / unused pixels) drawn pure white

    for file in files:
        data, pdcs = read_data(file)
        grid = format_data(data, pdcs)

        fig, ax = plt.subplots()
        im = ax.imshow(grid, origin="lower", cmap=cmap, norm=LogNorm(vmin=vmin, vmax=vmax))
        plt.colorbar(im, label="TCR")

        ax.axvline(64-0.5, color="black", linewidth=1)
        ax.axhline(64-0.5, color="black", linewidth=1)
        ax.set_xticks([])
        ax.set_yticks([])
        ax.set_title(f"Pixel TCR {file.parent.name}")

        plt.tight_layout()
        if save:
            savepath = file.with_suffix(".png")
            plt.savefig(savepath)
            print(f"File saved to {savepath}")
        if show:
            plt.show()
        plt.close()

draw_heatmaps(files, True, False, vrange=(1e1, 8e6))

Running file C:\Users\p6k\Desktop\UCSDPDCv2\data\Archive\TCR_5000nsHold_BiasSweep_LEDOff_RT\21.5V\20261020_09h20m50_TCR_H54_200ms_21V5_PDC0_PDC1_PDC2_PDC3.csv
File saved to C:\Users\p6k\Desktop\UCSDPDCv2\data\Archive\TCR_5000nsHold_BiasSweep_LEDOff_RT\21.5V\20261020_09h20m50_TCR_H54_200ms_21V5_PDC0_PDC1_PDC2_PDC3.png
Running file C:\Users\p6k\Desktop\UCSDPDCv2\data\Archive\TCR_5000nsHold_BiasSweep_LEDOff_RT\22.5V\20261020_10h08m13_TCR_H54_200ms_22V5_PDC0_PDC1_PDC2_PDC3.csv
File saved to C:\Users\p6k\Desktop\UCSDPDCv2\data\Archive\TCR_5000nsHold_BiasSweep_LEDOff_RT\22.5V\20261020_10h08m13_TCR_H54_200ms_22V5_PDC0_PDC1_PDC2_PDC3.png
Running file C:\Users\p6k\Desktop\UCSDPDCv2\data\Archive\TCR_5000nsHold_BiasSweep_LEDOff_RT\22V\20261020_09h44m17_TCR_H54_200ms_22V_PDC0_PDC1_PDC2_PDC3.csv
File saved to C:\Users\p6k\Desktop\UCSDPDCv2\data\Archive\TCR_5000nsHold_BiasSweep_LEDOff_RT\22V\20261020_09h44m17_TCR_H54_200ms_22V_PDC0_PDC1_PDC2_PDC3.png
Running file C:\Users\p6k\Desktop\UCSDPDCv2\data\